# Lab 5.4 — Measuring spin

**Goal:** simulate silver atoms passing through chains of Stern–Gerlach magnets, reproduce the famous Z → X → Z surprise, and check the rule

$$P(\text{up}) = \cos^2(\theta/2)$$

for a magnet tilted by $\theta$ from the direction the spin was last measured.

We describe a spin's state by a direction (a unit vector). Measuring along an axis $\hat a$ gives **up** with probability $(1 + \hat s \cdot \hat a)/2$, and afterwards the spin points along $+\hat a$ (up) or $-\hat a$ (down).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=7)

## 1. Directions and one measurement

A magnet tilted by `theta` degrees from +z toward +x measures along this axis:

In [ ]:
def axis(theta_deg):
    t = np.radians(theta_deg)
    return np.array([np.sin(t), 0.0, np.cos(t)])

def random_direction():
    # An unpolarised atom from the oven: any direction is equally likely
    z = rng.uniform(-1, 1)
    phi = rng.uniform(0, 2 * np.pi)
    r = np.sqrt(1 - z**2)
    return np.array([r * np.cos(phi), r * np.sin(phi), z])

def measure(state, a):
    # Returns (up, new_state)
    p_up = (1 + state @ a) / 2
    up = rng.random() < p_up
    return up, (a if up else -a)

Z, X = axis(0), axis(90)
ups = sum(measure(random_direction(), Z)[0] for _ in range(10_000))
print(f"One Z magnet: {ups / 10_000:.1%} up")

## 2. Chains of magnets

`run_chain` sends atoms through several magnets. After every magnet except the last, the **down** beam is blocked. It returns the fraction of surviving atoms that come out up from the last magnet.

In [ ]:
def run_chain(axes, n_atoms=20_000):
    up_count = reached = 0
    for _ in range(n_atoms):
        state = random_direction()
        for k, a in enumerate(axes):
            up, state = measure(state, a)
            last = k == len(axes) - 1
            if not last and not up:
                break                 # blocked
            if last:
                reached += 1
                up_count += up
    return up_count / reached, reached / n_atoms

for name, axes in [("Z, Z", [Z, Z]), ("Z, X", [Z, X]), ("Z, X, Z", [Z, X, Z])]:
    frac_up, frac_reached = run_chain(axes)
    print(f"{name:8s}  up at the end: {frac_up:6.1%}   reached the screen: {frac_reached:6.1%}")

Z, X, Z gives about 50% **down**, even though every down atom was blocked after the first magnet. The X measurement erased the z information.

## 3. The tilted magnet: checking cos²(θ/2)

In [ ]:
thetas = np.arange(0, 181, 15)
measured = [run_chain([Z, axis(t)], n_atoms=6_000)[0] for t in thetas]
theory = np.cos(np.radians(thetas) / 2) ** 2

plt.figure(figsize=(7, 4))
plt.plot(thetas, theory, label="cos²(θ/2)")
plt.plot(thetas, measured, "o", label="simulated atoms")
plt.xlabel("Tilt of second magnet from z (degrees)")
plt.ylabel("Fraction up")
plt.xticks(range(0, 181, 30))
plt.legend(); plt.grid(alpha=0.3)
plt.title("Spin measured along a tilted axis")
plt.show()

for t, m in zip(thetas, measured):
    if t in (0, 60, 90, 120, 180):
        print(f"θ = {t:3d}°: simulated {m:.3f}, theory {np.cos(np.radians(t)/2)**2:.3f}")

## 4. Newton vs quantum

A classical tiny magnet pointing in direction $\hat\mu$ is pushed in proportion to $\hat\mu\cdot\hat z$, which can be anything from −1 to +1. A quantum spin gives only +1 or −1. Compare the two screens:

In [ ]:
N = 20_000
classical = np.array([random_direction()[2] for _ in range(N)])
quantum = np.array([1 if measure(random_direction(), Z)[0] else -1 for _ in range(N)])

fig, axs = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
axs[0].hist(classical, bins=40, range=(-1.2, 1.2))
axs[0].set_title("Classical magnets: a smear")
axs[1].hist(quantum + rng.normal(0, 0.04, N), bins=40, range=(-1.2, 1.2))
axs[1].set_title("Quantum spin: two spots")
for ax in axs:
    ax.set_xlabel("Deflection along z")
axs[0].set_ylabel("Atoms")
plt.tight_layout(); plt.show()

## Exercises

1. **[Core]** Change `run_chain` to block the **up** beam instead. Do Z, Z and Z, X, Z still behave the same way?
2. **[Extension]** Send atoms through Z, then a magnet at 45°, then X. What fraction comes out +x at the end? Predict it first with cos²(θ/2) applied twice.
3. **[Challenge]** Put 10 magnets in a row, each tilted 18° more than the last (0°, 18°, …, 180°), blocking down after each. What fraction of atoms that reach the end come out up, and what fraction of all atoms survive? Compare with blocking after just one 180° magnet.